# Importing the Libraries

In [44]:
import requests
from bs4 import BeautifulSoup
import re
import time
import pandas as pd
import random
import numpy as np
import warnings
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import os
import undetected_chromedriver as uc

In [45]:
warnings.filterwarnings("ignore", category=FutureWarning)

# Creating the variables and functions

In [46]:
# The following function is meant to get the value from some cells where it comes like 50.5%

# Pandas does not recognise the number when it comes with %, since it's not interpreted as a number

def clean_value(val):
    if isinstance(val, str):
        val = val.strip()
        if val.endswith('%'):
            try:
                return float(val.replace('%', '')) / 100
            except:
                return None
        try:
            return float(val)
        except:
            return None
    return val

In [47]:
# rolling_cagr function:
# Calculates the Compound Annual Growth Rate (CAGR) in a rolling manner for each time point,
# always looking back over a 5-year window. If there is not enough data available, it returns NaN.
# This generates a full row of year-by-year CAGR values instead of a single aggregated growth figure.

def rolling_cagr(series, window=5):
    series = series.sort_index().astype(float)
    dates = series.index
    cagr_values = []

    for i in range(len(series)):
        if i < window:
            cagr_values.append(0)
            continue
        start = series.iloc[i - window]
        end = series.iloc[i]
        if pd.notna(start) and pd.notna(end) and start > 0 and end > 0:
            cagr = (end / start) ** (1 / window) - 1
        else:
            cagr = 0  # changed from np.nan
        cagr_values.append(cagr)

    return pd.Series(cagr_values, index=dates)


In [48]:
# This can be modified as needed. They are the ticket codes from the companies the user wants to analyse.

"""tickers = ['CBA.AX', 'NAB.AX', 'WBC.AX', 'MQG.AX', 'PNI.AX', 'HUB.AX', 'BHP.AX', 
           'RIO.AX', 'S32.AX', 'FMG.AX', 'LYC.AX', 'PLS.AX', 'CSL.AX', 'SHL.AX', 
           'ANN.AX', 'COH.AX', 'PME.AX', 'NAN.AX', 'WOW.AX', 'COL.AX', 'EDV.AX', 
           'A2M.AX', 'SHV.AX', 'WTC.AX', 'XRO.AX', 'TNE.AX', APX.AX', '360.AX', 
           'OCL.AX', 'CAR.AX', 'CPU.AX', 'IRE.AX', 'BRG.AX']"""

"tickers = ['CBA.AX', 'NAB.AX', 'WBC.AX', 'MQG.AX', 'PNI.AX', 'HUB.AX', 'BHP.AX', \n           'RIO.AX', 'S32.AX', 'FMG.AX', 'LYC.AX', 'PLS.AX', 'CSL.AX', 'SHL.AX', \n           'ANN.AX', 'COH.AX', 'PME.AX', 'NAN.AX', 'WOW.AX', 'COL.AX', 'EDV.AX', \n           'A2M.AX', 'SHV.AX', 'WTC.AX', 'XRO.AX', 'TNE.AX', APX.AX', '360.AX', \n           'OCL.AX', 'CAR.AX', 'CPU.AX', 'IRE.AX', 'BRG.AX']"

In [61]:
# This can be modified as needed. They are the ticket codes from the companies the user wants to analyse.

tickers = ['OCL.AX', 'HUB.AX', 'DTL.AX', 'IRE.AX', 'BVS.AX', 'PPS.AX', 'TNE.AX', # tech
           'CDA.AX', 'PFP.AX', 'BRG.AX', 'PMV.AX', 'LOV.AX', # consumer staples
           'RMD.AX', 'CSL.AX', 'SHL.AX', # health care
           'GQG.AX', 'QBE.AX', 'HMC.AX', 'PNI.AX', 'WAM.AX', # Financials
           'TEA.AX', 'VNT.AX', 'NWH.AX', 'BXB.AX', 'SRG.AX'] # Industrials

# Getting the data

## Company Info

In [50]:
# Start undetected ChromeDriver with standard options
options = uc.ChromeOptions()
# Uncomment below if you want visible browser (recommended for CAPTCHA sites)
# options.add_argument("--headless")
options.add_argument("--no-sandbox")
options.add_argument("--disable-gpu")

# Initialize the undetected driver
driver = uc.Chrome(options=options)

# Target URL to scrape
url = "https://www.marketindex.com.au/asx-listed-companies"
driver.get(url)

# Give time for the page to load fully
time.sleep(10)  # Increase if needed

# Capture the rendered HTML
html = driver.page_source

# Parse with BeautifulSoup
soup = BeautifulSoup(html, "html.parser")

# Find the main table by its class
table = soup.find('table', class_='mi-table mt-6')

# Extract the table into a DataFrame
df_asx_sectors = pd.read_html(str(table))[0]

# Keep only relevant columns
df_asx_sectors = df_asx_sectors[["Code", "Company", "Sector"]]

# Close the browser session
driver.quit()

# Load the official ASX listing (GICS industry group)
url = "https://www.asx.com.au/asx/research/ASXListedCompanies.csv"
df_asx_industries = pd.read_csv(url, skiprows=1)

# Standardize column for merging
df_asx_industries.rename(columns={"ASX code": "Code"}, inplace=True)

# Merge sectors (from Market Index) with industries (from ASX)
df_asx_sector_industry = df_asx_sectors.merge(
    df_asx_industries[["Code", "GICS industry group"]],
    on="Code",
    how="left"
)

## Exceptions to fix:

df_asx_sector_industry.loc[df_asx_sector_industry['Code'] == 'TEA', 'Sector'] = 'Industrials'

df_asx_sector_industry.loc[df_asx_sector_industry['Code'] == 'WAM', 'GICS industry group'] = 'Financial Services'

# Final merged DataFrame
df_asx_sector_industry

,Code,Company,Sector,GICS industry group
0,CBA,CBACommonwealth Bank of Australia,Financials,Banks
1,BHP,BHPBHP Group Ltd,Materials,Materials
2,NAB,NABNational Australia Bank Ltd,Financials,Banks
3,WBC,WBCWestpac Banking Corporation,Financials,Banks
4,CSL,CSLCSL Ltd,Health Care,"Pharmaceuticals, Biotechnology & Life Sciences"
...,...,...,...,...
2322,TTA,TTATTA Holdings Ltd,Consumer Discretionary,Consumer Discretionary Distribution & Retail
2323,CINPA,CINPACarlton Investments Ltd,NaN,NaN
2324,MIOR,MIORMacarthur Minerals Ltd,NaN,NaN
2325,SSLPA,SSLPASietel Ltd,NaN,NaN


In [62]:
company_info = {}

for ticker in tickers:
    # Separate company code and market suffix
    if "." in ticker:
        code, market = ticker.split(".")
    else:
        code, market = ticker, None

    # Initialize sector and industry as None
    sector = industry = None

    # If the company is from the ASX (Australia)
    if market == "AX":
        result = df_asx_sector_industry[df_asx_sector_industry["Code"] == code]
        if not result.empty:
            sector = result.iloc[0]["Sector"]
            industry = result.iloc[0]["GICS industry group"]
    else:
        # Placeholder for future support (e.g., US market)
        continue

    # Store results in dictionary
    company_info[ticker] = {"Sector": sector, "Industry": industry}


## Income

Now it's time to get the information about the income of each company.

It iterates through the ticket codes, requests the HTML of each, and then treats the data accordingly.

In [63]:
df_income = pd.DataFrame()

income_kpi_frames = {}

for ticker in tickers:
    
    url = f"https://discountingcashflows.com/company/{ticker}/income-statement/"
    headers = {"User-Agent": "Mozilla/5.0"}
    response = requests.get(url, headers=headers)

    soup = BeautifulSoup(response.text, 'html.parser')

    # Extract title and isolate the company name
    title = soup.title.string if soup.title else ""

    # Regex: capture everything before the ticker (inside parentheses). E.g.: 'Harvey Norman Holdings (HVN.AX) Income Annual - Discounting Cash Flows'
    # So it gets Harvey Norman Holdings
    match = re.match(rf"^(.*?)\s+\({re.escape(ticker)}\)", title)
    company_name = match.group(1).strip() if match else None

    # Find the table
    table = soup.find('table')

    # Convert HTML table to DataFrame
    df = pd.read_html(str(table))[0]
    
    # Drop LTM column (coming as NaN)
    df = df.drop(columns=["LTM  (Last Twelve Months)"])

    # Drop columns where name contains 'Unnamed'
    df = df.loc[:, ~df.columns.str.contains("^Unnamed")]

    # Simply drop the first row (wrong values like 'Report Filing' etc.)
    df = df.drop(index=0).reset_index(drop=True)

    # Now rename the first column to "Report Filing" for consistency
    df = df.rename(columns={df.columns[0]: "Report Filing"})

    # Reset index
    df = df.reset_index(drop=True)
    
    # Set the first column as the index (this column has metric names)
    df_copy = df.set_index("Report Filing")
    
    df_copy = df_copy.apply(pd.to_numeric, errors="coerce")  # Ensure numeric values
    

# --------------------------------------------------------- Calculating the necessary margins ---------------------------------------------------------

    ## REVENUE GROWTH

    insert_pos = df_copy.index.get_loc("Revenue") + 1

    df_copy = pd.concat([
    df_copy.iloc[:insert_pos],
    pd.DataFrame([df_copy.loc["Revenue"][::-1].pct_change()[::-1]], index=["Revenue Growth"]),
    df_copy.iloc[insert_pos:]
                        ])

    ## REVENUE GROWTH (CAGR 5y)

    if "Revenue" in df_copy.index:
        insert_pos = df_copy.index.get_loc("Revenue") + 1
        revenue_growth_series = rolling_cagr(df_copy.loc["Revenue"])
        df_copy = pd.concat([
            df_copy.iloc[:insert_pos],
            pd.DataFrame([revenue_growth_series], index=["Revenue Growth (CAGR 5y)"]),
            df_copy.iloc[insert_pos:]
        ])
    
    ## EBITDA MARGIN

    insert_pos = df_copy.index.get_loc("EBITDA") + 1
    
    df_copy = pd.concat([
        df_copy.iloc[:insert_pos],
        pd.DataFrame([df_copy.loc["EBITDA"] / df_copy.loc["Revenue"]], index=["EBITDA Margin"]),
        df_copy.iloc[insert_pos:]
    ])


    ## OPERATING EXPENSE RATIO

    insert_pos = df_copy.index.get_loc("Operating Expenses") + 1
    
    df_copy = pd.concat([
        df_copy.iloc[:insert_pos],
        pd.DataFrame([df_copy.loc["Operating Expenses"] / df_copy.loc["Revenue"].replace(0, np.nan)], index=["Operating Expense Ratio"]),
        df_copy.iloc[insert_pos:]
    ])


    ## NET INCOME RATIO

    insert_pos = df_copy.index.get_loc("Net Income") + 1
    df_copy = pd.concat([
        df_copy.iloc[:insert_pos],
        pd.DataFrame([df_copy.loc["Net Income"][::-1].pct_change()[::-1]], index=["Net Income Growth"]),
        df_copy.iloc[insert_pos:]
    ])

    ## EBITDA GROWTH (CAGR 5y + MoM)
    if "EBITDA" in df_copy.index:
        insert_pos = df_copy.index.get_loc("EBITDA") + 1
    
        # Calculates 5-year Compound Annual Growth Rate (CAGR)
        ebitda_cagr = rolling_cagr(df_copy.loc["EBITDA"])
        df_copy = pd.concat([
            df_copy.iloc[:insert_pos],
            pd.DataFrame([ebitda_cagr], index=["EBITDA Growth (CAGR 5y)"]),
            df_copy.iloc[insert_pos:]
        ])
    
        # Updates insertion position again
        insert_pos += 1
    
        # Calculates reverse percentage growth to preserve original chronological order
        ebitda_mom = df_copy.loc["EBITDA"][::-1].pct_change()[::-1]
        df_copy = pd.concat([
            df_copy.iloc[:insert_pos],
            pd.DataFrame([ebitda_mom], index=["EBITDA Growth"]),
            df_copy.iloc[insert_pos:]
        ])


    ## EPS GROWTH (CAGR 5y + MoM), usando EPS real
    if "Diluted Earnings Per Share" in df_copy.index:
        insert_pos = df_copy.index.get_loc("Diluted Earnings Per Share") + 1
    
        # Calculates 5-year Compound Annual Growth Rate (CAGR)
        eps_cagr = rolling_cagr(df_copy.loc["Diluted Earnings Per Share"])
        df_copy = pd.concat([
            df_copy.iloc[:insert_pos],
            pd.DataFrame([eps_cagr], index=["EPS Growth (CAGR 5y)"]),
            df_copy.iloc[insert_pos:]
        ])
    
        # Updates insertion position again
        insert_pos += 1
    
        # Calculates reverse percentage growth to preserve original chronological order
        eps_mom = df_copy.loc["Diluted Earnings Per Share"][::-1].pct_change()[::-1]
        df_copy = pd.concat([
            df_copy.iloc[:insert_pos],
            pd.DataFrame([eps_mom], index=["EPS Growth"]),
            df_copy.iloc[insert_pos:]
        ])



    ## GROSS PROFIT GROWTH (CAGR 5y) and GROSS PROFIT GROWTH
    if "Gross Profit" in df_copy.index:
        insert_pos = df_copy.index.get_loc("Gross Profit") + 1
        gp_growth = rolling_cagr(df_copy.loc["Gross Profit"])
        df_copy = pd.concat([
            df_copy.iloc[:insert_pos],
            pd.DataFrame([gp_growth], index=["Gross Profit Growth (CAGR 5y)"]),
            df_copy.iloc[insert_pos:]
        ])
        
        # Updates insertion position again
        insert_pos += 1
    
        # Calculates reverse percentage growth to preserve original chronological order
        gp_growth_mom = df_copy.loc["Gross Profit"][::-1].pct_change()[::-1]
        df_copy = pd.concat([
            df_copy.iloc[:insert_pos],
            pd.DataFrame([gp_growth_mom], index=["Gross Profit Growth"]),
            df_copy.iloc[insert_pos:]
        ])

# --------------------------------------------------------- END ---------------------------------------------------------

    #  Clean up infinities to avoid Power BI issues
    df_copy.replace([np.inf, -np.inf], np.nan, inplace=True)

    # Fix the broken index name
    df_copy.index.name = "Report Filing"
    
    # Extract only needed metrics for KPI calculations
    kpi_rows = ["EBITDA", "Revenue", "Weighted Average Shares Outstanding"]
    available_kpis = [row for row in kpi_rows if row in df_copy.index]

    if len(available_kpis) == len(kpi_rows):
        # Mini DataFrame just for KPIs
        df_kpi = df_copy.loc[kpi_rows].copy()

        # Rename for clarity
        ebitda = df_kpi.loc["EBITDA"]
        revenue = df_kpi.loc["Revenue"]
        shares = df_kpi.loc["Weighted Average Shares Outstanding"]

        # Calculations
        ebitda_per_share = ebitda / shares.replace(0, np.nan)
        ebitda_per_revenue = ebitda / revenue.replace(0, np.nan)

        # Append to the KPI frame
        df_kpi.loc["EBITDA Per Share"] = ebitda_per_share
        df_kpi.loc["EBITDA Per Revenue"] = ebitda_per_revenue

    # Save to dictionary (in wide format)
    income_kpi_frames[ticker] = df_kpi
    
    # Add a line number based on row order
    df_copy["Line"] = range(1, len(df_copy) + 1)

    # Then melt the dataframe
    df_melted = df_copy.reset_index().melt(id_vars=[df_copy.index.name, "Line"], 
                                       var_name="Date", 
                                       value_name="Value")

    # Rename the metric column
    df_melted = df_melted.rename(columns={df_copy.index.name: "Metric"})
    df_melted["Date"] = pd.to_datetime(df_melted["Date"], errors="coerce")
    df_melted["Value"] = pd.to_numeric(df_melted["Value"], errors="coerce")

    # Add ticker and company name as first columns
    df_melted.insert(0, "Ticker", ticker)
    df_melted.insert(1, "Company", company_name)
    df_melted.insert(2, "Report_Type", "Income")
    df_melted.insert(3, "Sector", company_info[ticker]["Sector"])
    df_melted.insert(4, "Industry", company_info[ticker]["Industry"])
    
    # Finally, it joins the created df with the existing df_income
    df_income = pd.concat([df_income, df_melted], ignore_index=True)

    
    if len(tickers) > 1:
        time.sleep(random.uniform(4, 10))

In [64]:
df_income

,Ticker,Company,Report_Type,Sector,Industry,Metric,Line,Date,Value
0,OCL.AX,Objective Corporation Limited,Income,Information Technology,Software & Services,Revenue,1,2024-06-30,114.800000
1,OCL.AX,Objective Corporation Limited,Income,Information Technology,Software & Services,Revenue Growth (CAGR 5y),2,2024-06-30,0.130905
2,OCL.AX,Objective Corporation Limited,Income,Information Technology,Software & Services,Revenue Growth,3,2024-06-30,0.054178
3,OCL.AX,Objective Corporation Limited,Income,Information Technology,Software & Services,Cost of Revenue,4,2024-06-30,7.600000
4,OCL.AX,Objective Corporation Limited,Income,Information Technology,Software & Services,Gross Profit,5,2024-06-30,107.200000
...,...,...,...,...,...,...,...,...,...
18079,SRG.AX,SRG Global Limited,Income,Industrials,Capital Goods,Diluted Earnings Per Share,29,2006-06-30,2.880000
18080,SRG.AX,SRG Global Limited,Income,Industrials,Capital Goods,EPS Growth (CAGR 5y),30,2006-06-30,0.000000
18081,SRG.AX,SRG Global Limited,Income,Industrials,Capital Goods,EPS Growth,31,2006-06-30,NaN
18082,SRG.AX,SRG Global Limited,Income,Industrials,Capital Goods,Weighted Average Shares Outstanding,32,2006-06-30,0.653000


## Balance Sheet

In [65]:
df_balance_sheet = pd.DataFrame()

for ticker in tickers:

    url = f"https://discountingcashflows.com/company/{ticker}/balance-sheet-statement/"
    headers = {"User-Agent": "Mozilla/5.0"}
    response = requests.get(url, headers=headers)

    soup = BeautifulSoup(response.text, 'html.parser')

    # Extract title and isolate the company name
    title = soup.title.string if soup.title else ""

    # Regex: capture everything before the ticker (inside parentheses). E.g.: 'Harvey Norman Holdings (HVN.AX) Income Annual - Discounting Cash Flows'
    # So it gets Harvey Norman Holdings
    
    match = re.match(rf"^(.*?)\s+\({re.escape(ticker)}\)", title)
    company_name = match.group(1).strip() if match else None

    # Find the table
    table = soup.find('table')

    # Convert HTML table to DataFrame
    df = pd.read_html(str(table))[0]

    # Drop columns where name contains 'Unnamed'
    df = df.loc[:, ~df.columns.str.contains("^Unnamed")]

    # Simply drop the first row (wrong values like 'Report Filing' etc.)
    df = df.drop(index=0).reset_index(drop=True)

    # Now rename the first column to "Report Filing" for consistency
    df = df.rename(columns={df.columns[0]: "Report Filing"})

    # Reset index
    df = df.reset_index(drop=True)

    # Set the first column as the index (this column has metric names)
    df_copy = df.set_index(df.columns[0])

# --------------------------------------------------------- Calculating the necessary margins ---------------------------------------------------------

    # Ensure columns are numeric
    df_copy = df_copy.apply(pd.to_numeric, errors="coerce")

    df_copy = df_copy.round(3)
    
    # 1. Cash to Assets
    insert_pos = df_copy.index.get_loc("Cash and Short Term Investments") + 1
    df_copy = pd.concat([
        df_copy.iloc[:insert_pos],
        pd.DataFrame([df_copy.loc["Cash and Short Term Investments"] / df_copy.loc["Total Assets"]], index=["Cash to Assets"]),
        df_copy.iloc[insert_pos:]
    ])
    
    # 2. Net Debt / EBITDA (requires income_kpi_frames)
    if ticker in income_kpi_frames and "EBITDA" in income_kpi_frames[ticker].index:
        insert_pos = df_copy.index.get_loc("Net Debt") + 1
        ebitda = income_kpi_frames[ticker].loc["EBITDA"]
        df_copy = pd.concat([
            df_copy.iloc[:insert_pos],
            pd.DataFrame([df_copy.loc["Net Debt"] / ebitda.replace(0, np.nan)], index=["Net Debt to EBITDA"]),
            df_copy.iloc[insert_pos:]
        ])
    
    # 3. Intangibles to Assets
    insert_pos = df_copy.index.get_loc("Goodwill and Intangible Assets") + 1
    df_copy = pd.concat([
        df_copy.iloc[:insert_pos],
        pd.DataFrame([df_copy.loc["Goodwill and Intangible Assets"] / df_copy.loc["Total Assets"]], index=["Intangibles to Assets"]),
        df_copy.iloc[insert_pos:]
    ])
    
    # 4. Tangible Book Value
    insert_pos = df_copy.index.get_loc("Total Stockholders' Equity") + 1
    df_copy = pd.concat([
        df_copy.iloc[:insert_pos],
        pd.DataFrame([df_copy.loc["Total Stockholders' Equity"] - df_copy.loc["Goodwill and Intangible Assets"]], index=["Tangible Book Value"]),
        df_copy.iloc[insert_pos:]
    ])
    
    # 5. Inventory to Assets
    if "Inventory" in df_copy.index:
        insert_pos = df_copy.index.get_loc("Inventory") + 1
        df_copy = pd.concat([
            df_copy.iloc[:insert_pos],
            pd.DataFrame([df_copy.loc["Inventory"] / df_copy.loc["Total Assets"]], index=["Inventory to Assets"]),
            df_copy.iloc[insert_pos:]
        ])
    
    # 6. Receivables to Assets
    if "Receivables" in df_copy.index:
        insert_pos = df_copy.index.get_loc("Receivables") + 1
        df_copy = pd.concat([
            df_copy.iloc[:insert_pos],
            pd.DataFrame([df_copy.loc["Receivables"] / df_copy.loc["Total Assets"]], index=["Receivables to Assets"]),
            df_copy.iloc[insert_pos:]
        ])
    
    # 7. Fixed Assets to Total Assets
    insert_pos = df_copy.index.get_loc("Property, Plant and Equipment") + 1
    df_copy = pd.concat([
        df_copy.iloc[:insert_pos],
        pd.DataFrame([df_copy.loc["Property, Plant and Equipment"] / df_copy.loc["Total Assets"]], index=["Fixed Assets to Total Assets"]),
        df_copy.iloc[insert_pos:]
    ])
    
    # 8. Crescimento do Equity (Equity Growth)
    insert_pos = df_copy.index.get_loc("Total Stockholders' Equity") + 1
    df_copy = pd.concat([
        df_copy.iloc[:insert_pos],
        pd.DataFrame([df_copy.loc["Total Stockholders' Equity"][::-1].pct_change()[::-1]], index=["Equity Growth"]),
        df_copy.iloc[insert_pos:]
    ])
    
    # 9. Crescimento Retained Earnings (Retained Earnings Growth)
    insert_pos = df_copy.index.get_loc("Retained Earnings") + 1
    df_copy = pd.concat([
        df_copy.iloc[:insert_pos],
        pd.DataFrame([df_copy.loc["Retained Earnings"][::-1].pct_change()[::-1]], index=["Retained Earnings Growth"]),
        df_copy.iloc[insert_pos:]
    ])

    # 10. Net Debt / Equity (requires income_kpi_frames)
    insert_pos = df_copy.index.get_loc("Net Debt") + 1
    df_copy = pd.concat([
        df_copy.iloc[:insert_pos],
        pd.DataFrame([df_copy.loc["Net Debt"] / df_copy.loc["Total Equity"].replace(0, np.nan)], index=["Net Debt to Equity"]),
        df_copy.iloc[insert_pos:]
    ])
        
    #  Restore index name (required for melt to work)
    df_copy.index.name = "Report Filing"
    
# --------------------------------------------------------- END ---------------------------------------------------------

    #  Clean up infinities to avoid Power BI issues
    df_copy.replace([np.inf, -np.inf], np.nan, inplace=True)

    # Add a line number based on row order
    df_copy["Line"] = range(1, len(df_copy) + 1)

    # Then melt the dataframe
    df_melted = df_copy.reset_index().melt(id_vars=[df_copy.index.name, "Line"], 
                                       var_name="Date", 
                                       value_name="Value")

    # Rename the metric column
    df_melted = df_melted.rename(columns={df_copy.index.name: "Metric"})

    # Optional cleanup
    df_melted["Date"] = pd.to_datetime(df_melted["Date"], errors="coerce")
    df_melted["Value"] = pd.to_numeric(df_melted["Value"], errors="coerce")

    # Add ticker and company name as first columns
    df_melted.insert(0, "Ticker", ticker)
    df_melted.insert(1, "Company", company_name)
    df_melted.insert(2, "Report_Type", "Balance Sheet")
    df_melted.insert(3, "Sector", company_info[ticker]["Sector"])
    df_melted.insert(4, "Industry", company_info[ticker]["Industry"])
    
    # Finally, it joins the created df with the existing df_income
    df_balance_sheet = pd.concat([df_balance_sheet, df_melted], ignore_index=True)
    
    if len(tickers) > 1:
        time.sleep(random.uniform(4, 10))

In [66]:
df_balance_sheet

,Ticker,Company,Report_Type,Sector,Industry,Metric,Line,Date,Value
0,OCL.AX,Objective Corporation Limited,Balance Sheet,Information Technology,Software & Services,Total Current Assets,1,2024-06-30,105.900000
1,OCL.AX,Objective Corporation Limited,Balance Sheet,Information Technology,Software & Services,Cash and Short Term Investments,2,2024-06-30,95.980000
2,OCL.AX,Objective Corporation Limited,Balance Sheet,Information Technology,Software & Services,Cash to Assets,3,2024-06-30,0.555119
3,OCL.AX,Objective Corporation Limited,Balance Sheet,Information Technology,Software & Services,Cash & Equivalents,4,2024-06-30,95.980000
4,OCL.AX,Objective Corporation Limited,Balance Sheet,Information Technology,Software & Services,Short Term Investments,5,2024-06-30,0.000000
...,...,...,...,...,...,...,...,...,...
28491,SRG.AX,SRG Global Limited,Balance Sheet,Industrials,Capital Goods,Total Investments,48,2006-06-30,-0.000000
28492,SRG.AX,SRG Global Limited,Balance Sheet,Industrials,Capital Goods,Total Debt,49,2006-06-30,2.490000
28493,SRG.AX,SRG Global Limited,Balance Sheet,Industrials,Capital Goods,Net Debt,50,2006-06-30,2.490000
28494,SRG.AX,SRG Global Limited,Balance Sheet,Industrials,Capital Goods,Net Debt to Equity,51,2006-06-30,0.555804


## Cash Flow

In [67]:
df_cash_flow = pd.DataFrame()

for ticker in tickers:

    url = f"https://discountingcashflows.com/company/{ticker}/cash-flow-statement/"
    headers = {"User-Agent": "Mozilla/5.0"}
    response = requests.get(url, headers=headers)

    soup = BeautifulSoup(response.text, 'html.parser')

    # Extract title and isolate the company name
    title = soup.title.string if soup.title else ""

    # Regex: capture everything before the ticker (inside parentheses). E.g.: 'Harvey Norman Holdings (HVN.AX) Income Annual - Discounting Cash Flows'
    # So it gets Harvey Norman Holdings
    
    match = re.match(rf"^(.*?)\s+\({re.escape(ticker)}\)", title)
    company_name = match.group(1).strip() if match else None

    # Find the table
    table = soup.find('table')

    # Convert HTML table to DataFrame
    df = pd.read_html(str(table))[0]

    # Drop columns where name contains 'Unnamed'
    df = df.loc[:, ~df.columns.str.contains("^Unnamed")]

    # Drop LTM column (coming as NaN)

    df = df.drop(columns=["LTM  (Last Twelve Months)"])

    # Simply drop the first row (wrong values like 'Report Filing' etc.)
    df = df.drop(index=0).reset_index(drop=True)

    # Now rename the first column to "Report Filing" for consistency
    df = df.rename(columns={df.columns[0]: "Report Filing"})

    # Reset index
    df = df.reset_index(drop=True)

    # Set the first column as the index (this column has metric names)
    df_copy = df.set_index(df.columns[0])

# --------------------------------------------------------- Calculating the necessary margins ---------------------------------------------------------

    # Ensure numeric values
    df_copy = df_copy.apply(pd.to_numeric, errors="coerce")
    
    # 1. CapEx to Revenue
    if ticker in income_kpi_frames and "Revenue" in income_kpi_frames[ticker].index:
        insert_pos = df_copy.index.get_loc("Capital Expenditure") + 1
        revenue = income_kpi_frames[ticker].loc["Revenue"]
        df_copy = pd.concat([
            df_copy.iloc[:insert_pos],
            pd.DataFrame([df_copy.loc["Capital Expenditure"] / revenue.replace(0, np.nan)], index=["CapEx to Revenue"]),
            df_copy.iloc[insert_pos:]
        ])
    
    # 2. Debt Repayment / Free Cash Flow
    if "Debt Repayment" in df_copy.index and "Free Cash Flow" in df_copy.index:
        insert_pos = df_copy.index.get_loc("Debt Repayment") + 1
        df_copy = pd.concat([
            df_copy.iloc[:insert_pos],
            pd.DataFrame([df_copy.loc["Debt Repayment"] / df_copy.loc["Free Cash Flow"].replace(0, np.nan)], index=["Debt Repayment / FCF"]),
            df_copy.iloc[insert_pos:]
        ])
    
    # 3. Dividends Paid / Free Cash Flow
    if "Dividends Paid" in df_copy.index and "Free Cash Flow" in df_copy.index:
        insert_pos = df_copy.index.get_loc("Dividends Paid") + 1
        df_copy = pd.concat([
            df_copy.iloc[:insert_pos],
            pd.DataFrame([df_copy.loc["Dividends Paid"] / df_copy.loc["Free Cash Flow"].replace(0, np.nan)], index=["Dividends Paid / FCF"]),
            df_copy.iloc[insert_pos:]
        ])
    
    # Restore index name so melt works
    df_copy.index.name = "Report Filing"

    #  Clean up infinities to avoid Power BI issues
    df_copy.replace([np.inf, -np.inf], np.nan, inplace=True)

# --------------------------------------------------------- END ---------------------------------------------------------

    # Add a line number based on row order
    df_copy["Line"] = range(1, len(df_copy) + 1)

    # Then melt the dataframe
    df_melted = df_copy.reset_index().melt(id_vars=[df_copy.index.name, "Line"], 
                                       var_name="Date", 
                                       value_name="Value")

    # Rename the metric column
    df_melted = df_melted.rename(columns={df_copy.index.name: "Metric"})

    # Optional cleanup
    df_melted["Date"] = pd.to_datetime(df_melted["Date"], errors="coerce")
    df_melted["Value"] = pd.to_numeric(df_melted["Value"], errors="coerce")

    # Add ticker and company name as first columns
    df_melted.insert(0, "Ticker", ticker)
    df_melted.insert(1, "Company", company_name)
    df_melted.insert(2, "Report_Type", "Cash Flow")
    df_melted.insert(3, "Sector", company_info[ticker]["Sector"])
    df_melted.insert(4, "Industry", company_info[ticker]["Industry"])
    
    # Finally, it joins the created df with the existing df_income
    df_cash_flow = pd.concat([df_cash_flow, df_melted], ignore_index=True)
    
    if len(tickers) > 1:
        time.sleep(random.uniform(4, 10))

## Ratios

In [68]:
df_ratios = pd.DataFrame()


for ticker in tickers:

    url = f"https://discountingcashflows.com/company/{ticker}/ratios/"
    headers = {"User-Agent": "Mozilla/5.0"}
    response = requests.get(url, headers=headers)

    soup = BeautifulSoup(response.text, 'html.parser')

    # Extract title and isolate the company name
    title = soup.title.string if soup.title else ""

    # Regex: capture everything before the ticker (inside parentheses). E.g.: 'Harvey Norman Holdings (HVN.AX) Income Annual - Discounting Cash Flows'
    # So it gets Harvey Norman Holdings
    
    match = re.match(rf"^(.*?)\s+\({re.escape(ticker)}\)", title)
    company_name = match.group(1).strip() if match else None

    # Find the table
    table = soup.find('table')

    if not table:
        print(f"⚠️  No table found for {ticker} (might be 500 content)")
        continue

    # Convert HTML table to DataFrame
    df = pd.read_html(str(table))[0]

    # Drop LTM column (coming as NaN)

    df = df.drop(columns=["LTM  (Last Twelve Months)"])

    # Drop columns where name contains 'Unnamed'
    df = df.loc[:, ~df.columns.str.contains("^Unnamed")]
    
    # Ensure wide numeric format for calculation
    for col in df.columns[1:]:
        df[col] = df[col].map(clean_value)
    
    # If income KPIs are available for this ticker
    if ticker in income_kpi_frames:
        kpis = income_kpi_frames[ticker]
        
        # Standardise: income_kpi_frames will adapt to Ratios columns
        all_dates = df.columns[1:]  # the columns after "Period Ending:"

        # updates the EBITDA Per Share and EBITDA Per Revenue value in the Ratios DF with the value calculated in the income dictionary
        for metric in ["EBITDA Per Share", "EBITDA Per Revenue"]:
            if metric in df["Period Ending:"].values and metric in kpis.index:
                kpi_values = kpis.loc[metric].reindex(all_dates)
                df.loc[df["Period Ending:"].astype(str).str.strip() == metric, all_dates] = kpi_values.values

        # Calculate EV to EBITDA only if both parts are present
        ev_row = df[df["Period Ending:"].str.strip() == "Enterprise Value Per Share"]
        eps_row = df[df["Period Ending:"].str.strip() == "EBITDA Per Share"]

        if not ev_row.empty and not eps_row.empty:
            ev_values = ev_row.iloc[0, 1:].astype(float)
            eps_values = eps_row.iloc[0, 1:].replace(0, np.nan).astype(float)
            ev_to_ebitda = ev_values / eps_values

            df.loc[df["Period Ending:"].astype(str).str.strip() == "EV to EBITDA", all_dates] = ev_to_ebitda.values

            
    # Now drop fully-NaN rows (excluding label column)
    df = df.dropna(how="all", subset=df.columns[1:])

    # Reset index
    df = df.reset_index(drop=True)

    # Set the first column as the index (this column has metric names)
    df_copy = df.set_index(df.columns[0])

# --------------------------------------------------------- Calculating the necessary margins ---------------------------------------------------------

    ## BOOK VALUE GROWTH (CAGR 5y) AND BOOK VALUE GROWTH

    if "Book Value Per Share" in df_copy.index:
        insert_pos = df_copy.index.get_loc("Book Value Per Share") + 1
        bv_growth = rolling_cagr(df_copy.loc["Book Value Per Share"])
        
        df_copy = pd.concat([
            df_copy.iloc[:insert_pos],
            pd.DataFrame([bv_growth], index=["Book Value Growth (CAGR 5y)"]),
            df_copy.iloc[insert_pos:]
        ])

        insert_pos += 1

        # Calculates reverse percentage growth to preserve original chronological order
        bv_growth_mom = df_copy.loc["Book Value Per Share"][::-1].pct_change()[::-1]
    
        df_copy = pd.concat([
            df_copy.iloc[:insert_pos],
            pd.DataFrame([bv_growth_mom], index=["Book Value Growth"]),
            df_copy.iloc[insert_pos:]
        ])

        
        

    # Restore index name so melt works
    df_copy.index.name = "Report Filing"

    #  Clean up infinities to avoid Power BI issues
    df_copy.replace([np.inf, -np.inf], np.nan, inplace=True)

# --------------------------------------------------------- END ---------------------------------------------------------

    # Add a line number based on row order
    df_copy["Line"] = range(1, len(df_copy) + 1)

    # Then melt the dataframe
    df_melted = df_copy.reset_index().melt(id_vars=[df_copy.index.name, "Line"], 
                                       var_name="Date", 
                                       value_name="Value")

    # Rename the metric column
    df_melted = df_melted.rename(columns={df_copy.index.name: "Metric"})

    # Optional cleanup
    df_melted["Date"] = pd.to_datetime(df_melted["Date"], errors="coerce")
    df_melted["Value"] = df_melted["Value"].map(clean_value)

    # Add ticker and company name as first columns
    df_melted.insert(0, "Ticker", ticker)
    df_melted.insert(1, "Company", company_name)
    df_melted.insert(2, "Report_Type", "Ratios")
    df_melted.insert(3, "Sector", company_info[ticker]["Sector"])
    df_melted.insert(4, "Industry", company_info[ticker]["Industry"])

    # Finally, it joins the created df with the existing df_income
    df_ratios = pd.concat([df_ratios, df_melted], ignore_index=True)
    
    
    ev_row = None
    eps_row = None
    ev_values = None
    eps_values = None
    ev_to_ebitda = None

    if len(tickers) > 1:
        time.sleep(random.uniform(4, 10))

In [69]:
df_ratios

,Ticker,Company,Report_Type,Sector,Industry,Metric,Line,Date,Value
0,OCL.AX,Objective Corporation Limited,Ratios,Information Technology,Software & Services,Price to Earnings Ratio,1,2024-06-30,36.30
1,OCL.AX,Objective Corporation Limited,Ratios,Information Technology,Software & Services,Price to Sales Ratio,2,2024-06-30,9.91
2,OCL.AX,Objective Corporation Limited,Ratios,Information Technology,Software & Services,Price to Book Ratio,3,2024-06-30,12.32
3,OCL.AX,Objective Corporation Limited,Ratios,Information Technology,Software & Services,Price to Free Cash Flow Ratio,4,2024-06-30,27.95
4,OCL.AX,Objective Corporation Limited,Ratios,Information Technology,Software & Services,Price to Operating Cash Flow Ratio,5,2024-06-30,20.39
...,...,...,...,...,...,...,...,...,...
30664,SRG.AX,SRG Global Limited,Ratios,Industrials,Capital Goods,Days of Inventory Outstanding,52,2006-06-30,3.42
30665,SRG.AX,SRG Global Limited,Ratios,Industrials,Capital Goods,Days of Payables Outstanding,53,2006-06-30,62.84
30666,SRG.AX,SRG Global Limited,Ratios,Industrials,Capital Goods,Cash Conversion Cycle,54,2006-06-30,47.85
30667,SRG.AX,SRG Global Limited,Ratios,Industrials,Capital Goods,Cash Conversion Ratio,55,2006-06-30,0.00


# Combining the DFs and Saving the Excel file

In [70]:
df_all = pd.concat([df_income, df_balance_sheet, df_cash_flow, df_ratios], ignore_index=True)

df_all

,Ticker,Company,Report_Type,Sector,Industry,Metric,Line,Date,Value
0,OCL.AX,Objective Corporation Limited,Income,Information Technology,Software & Services,Revenue,1,2024-06-30,114.800000
1,OCL.AX,Objective Corporation Limited,Income,Information Technology,Software & Services,Revenue Growth (CAGR 5y),2,2024-06-30,0.130905
2,OCL.AX,Objective Corporation Limited,Income,Information Technology,Software & Services,Revenue Growth,3,2024-06-30,0.054178
3,OCL.AX,Objective Corporation Limited,Income,Information Technology,Software & Services,Cost of Revenue,4,2024-06-30,7.600000
4,OCL.AX,Objective Corporation Limited,Income,Information Technology,Software & Services,Gross Profit,5,2024-06-30,107.200000
...,...,...,...,...,...,...,...,...,...
95910,SRG.AX,SRG Global Limited,Ratios,Industrials,Capital Goods,Days of Inventory Outstanding,52,2006-06-30,3.420000
95911,SRG.AX,SRG Global Limited,Ratios,Industrials,Capital Goods,Days of Payables Outstanding,53,2006-06-30,62.840000
95912,SRG.AX,SRG Global Limited,Ratios,Industrials,Capital Goods,Cash Conversion Cycle,54,2006-06-30,47.850000
95913,SRG.AX,SRG Global Limited,Ratios,Industrials,Capital Goods,Cash Conversion Ratio,55,2006-06-30,0.000000


In [71]:
df_all.to_excel("Stock Data.xlsx", index=False)